
# 01 — Parsing des données météorologiques JSON

## Objectif
Transformer les données météorologiques brutes de Météo-France en colonnes structurées et exploitables.

## Source
`dbo.raw_meteo_france` — Table Bronze du Lakehouse.

## Destination
`dbo.raw_meteo_france_parsing` — Table contenant les données JSON extraites.

## Traitements prévus
- Lecture des données Bronze.
- Extraction des champs JSON : latitude, longitude, température, humidité, vent et précipitations.
- Conversion des données en types adaptés.
- Enregistrement des résultats au format Delta.
- Vérification du nombre d'observations.

## Intégration au pipeline
Ce notebook sera exécuté après l'activité de copie Eventhouse → Bronze et avant le notebook de nettoyage Silver.


In [1]:

# ==================================================
# ÉTAPE 02 — LECTURE DES DONNÉES BRONZE
# ==================================================

# A — Lire les données Bronze depuis le Lakehouse
df_raw = spark.table("dbo.raw_meteo_france")

# B — Vérifier le nombre d'observations
print("Nombre de relevés Bronze :", df_raw.count())

# C — Vérifier la structure des données
df_raw.printSchema()

# D — Afficher les 5 premières observations
display(df_raw.limit(5))


StatementMeta(, 9dde3496-7be4-4895-837d-64076b5f59a9, 3, Finished, Available, Finished, False)

Nombre de relevés Bronze : 127130
root
 |-- source: string (nullable = true)
 |-- observation_date: timestamp (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- data: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 36b5f20b-a3ca-43be-bc56-c96b2a835571)

In [2]:

# ==================================================
# ÉTAPE 03 — EXTRACTION DES CHAMPS JSON
# ==================================================

from pyspark.sql import functions as F

# A — Extraire les coordonnées géographiques
df_parsing = (
    df_raw
    .withColumn("latitude", F.get_json_object("data", "$.lat").cast("double"))
    .withColumn("longitude", F.get_json_object("data", "$.lon").cast("double"))
    .withColumn("geo_id_insee", F.get_json_object("data", "$.geo_id_insee"))

    # B — Extraire les mesures météorologiques
    .withColumn("temperature_k", F.get_json_object("data", "$.t").cast("double"))
    .withColumn("humidite_pct", F.get_json_object("data", "$.u").cast("double"))
    .withColumn("vitesse_vent", F.get_json_object("data", "$.ff").cast("double"))
    .withColumn("direction_vent", F.get_json_object("data", "$.dd").cast("double"))
    .withColumn("precipitations", F.get_json_object("data", "$.rr_per").cast("double"))
)

# C — Vérifier le nombre d'observations
print("Nombre de relevés après parsing :", df_parsing.count())

# D — Afficher les données extraites
display(
    df_parsing.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_k",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(10)
)


StatementMeta(, 9dde3496-7be4-4895-837d-64076b5f59a9, 4, Finished, Available, Finished, False)

Nombre de relevés après parsing : 127130


SynapseWidget(Synapse.DataFrame, d66212cc-bae6-4a31-b6d8-2106c991122b)

In [3]:

# ==================================================
# ÉTAPE 04 — ENREGISTREMENT DES DONNÉES PARSÉES
# ==================================================

# A — Enregistrer les données au format Delta
df_parsing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbo.raw_meteo_france_parsing")

# B — Relire la table enregistrée
df_verification = spark.table("dbo.raw_meteo_france_parsing")

# C — Vérifier le nombre d'observations
print("Nombre de relevés enregistrés :", df_verification.count())

# D — Afficher un aperçu
display(
    df_verification.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_k",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(10)
)


StatementMeta(, 9dde3496-7be4-4895-837d-64076b5f59a9, 5, Finished, Available, Finished, False)

Nombre de relevés enregistrés : 127130


SynapseWidget(Synapse.DataFrame, a671f183-4473-411b-bad6-7a642df825ae)